In [1]:
import networkx as nx
from networkx.algorithms.community import louvain_communities
from networkx.algorithms.community.quality import modularity
import json
import matplotlib.pyplot as plt
from operator import itemgetter
from typing import Set, Dict, Any, Tuple
import os

import graph_creation
from operator import itemgetter # Utilisé pour trier

In [2]:
# --- Constants for file paths ---
DEFAULT_JSON_PATH = 'data/processed/unified_articles.json'
FILTERED_JSON_PATH = 'data/processed/filtered_articles.json'

## 1. Function to filter the JSON based on in-degree and save the reduced version

def filter_json_and_save(
    json_path: str = DEFAULT_JSON_PATH, 
    output_json_path: str = FILTERED_JSON_PATH,
    top_n: int = 10000
) -> bool:
    """
    Reduces the JSON file by keeping only the articles corresponding to the 
    'top_n' non-isolated nodes with the highest in-degree in the full graph.
    Saves the filtered JSON to 'output_json_path'.

    Returns True if filtering and saving were successful, False otherwise.
    """
    
    # 0. CHECK IF FILE ALREADY EXISTS
    if os.path.exists(output_json_path):
        print(f"✅ Filtered JSON file already exists at: {output_json_path}. Skipping filtering.")
        return True # Le fichier existe, on considère que c'est réussi.
        
    print(f"Starting JSON filtering based on in-degree (top {top_n})...")
    
    # 1. Load JSON data (Consolidated try/except)
    try:
        with open(json_path, 'r', encoding='utf-8') as json_file:
            data: Dict[str, Any] = json.load(json_file)
            articles: list = data.get('articles', [])
            
    except FileNotFoundError:
        print(f"Error: JSON file not found at: {json_path}")
        return False
    except json.JSONDecodeError:
        print(f"Error: Invalid JSON format in file: {json_path}")
        return False
    except Exception as e:
        print(f"Unexpected error during JSON loading: {e}")
        return False

    # 2. Create the full graph to calculate degrees
    print(f"Building initial full graph from {len(articles)} articles...")
    G_full = nx.DiGraph()
    article_id_to_data = {} 
    
    for article in articles:
        article_id = article.get('id')
        if article_id is not None:
            G_full.add_node(article_id) 
            article_id_to_data[article_id] = article
            for link in article.get('refs', []):
                G_full.add_edge(article_id, link)

    # 3. Select the top N nodes (via in-degree)
    print(f"Calculating in-degree and selecting the top {top_n} most popular nodes...")
    in_degrees = G_full.in_degree()
    sorted_nodes: list[Tuple[Any, int]] = sorted(in_degrees, key=itemgetter(1), reverse=True)
    top_node_ids: Set[Any] = {node for node, degree in sorted_nodes[:top_n]}

    # 4. Create the temporary subgraph restricted to the top N and filter isolated nodes
    G_temp = nx.DiGraph()
    G_temp.add_nodes_from(top_node_ids)
    
    for u, v in G_full.edges():
        if u in top_node_ids and v in top_node_ids:
            G_temp.add_edge(u, v)
            
    isolated_nodes = list(nx.isolates(G_temp))
    G_temp.remove_nodes_from(isolated_nodes)
    
    final_node_ids: Set[Any] = set(G_temp.nodes())
    
    print(f"Isolated nodes removed: {len(isolated_nodes)}")
    print(f"Nodes kept for filtering: {len(final_node_ids)}")

    # 5. Create and save the filtered JSON (Consolidated try/except)
    filtered_articles = [
        article_id_to_data[id] 
        for id in final_node_ids 
        if id in article_id_to_data
    ]
    data_filtered = {"articles": filtered_articles}
    
    print(f"Filtered articles to be saved: {len(filtered_articles)}")
    
    try:
        os.makedirs(os.path.dirname(output_json_path) or '.', exist_ok=True)
        with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(data_filtered, f, indent=4)
        print(f"Filtered JSON saved successfully to: {output_json_path}")
        return True
    except Exception as e:
        print(f"Error saving filtered JSON: {e}")
        return False


## 2. Function to build the graph from the filtered JSON (IN MEMORY ONLY)

def create_graph_from_filtered_json(
    json_path: str = FILTERED_JSON_PATH,
) -> nx.DiGraph:
    """
    Builds a directed graph (DiGraph) from the filtered JSON file.
    The graph is created in memory and is NOT saved or loaded from disk.
    """

    print("--- Creating graph from filtered JSON (in memory)... ---")
    
    G = nx.DiGraph()
    
    # 1. Load JSON data (Consolidated try/except)
    try:
        with open(json_path, 'r', encoding='utf-8') as json_file:
            data = json.load(json_file)
            
        articles = data.get('articles', [])
        
        # 2. Graph creation
        for article in articles:
            article_id = article.get('id')
            if article_id is not None:
                G.add_node(article_id) 
                for link in article.get('refs', []):
                    G.add_edge(article_id, link)
                    
        # 3. Safety check: Remove any isolated nodes
        isolated_nodes = list(nx.isolates(G))
        if isolated_nodes:
             print(f"Warning: {len(isolated_nodes)} isolated nodes found and removed.")
             G.remove_nodes_from(isolated_nodes)
             
    except FileNotFoundError:
        print(f"Error: Filtered JSON file not found at: {json_path}. Run 'filter_json_and_save' first.")
        return nx.DiGraph()
    except json.JSONDecodeError:
        print(f"Error: Invalid JSON format in file: {json_path}")
        return nx.DiGraph()
    except Exception as e:
        print(f"Error during graph creation: {e}")
        return nx.DiGraph()

    print(f"Graph created: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.")
    
    return G

In [3]:
filter_json_and_save()

✅ Filtered JSON file already exists at: data/processed/filtered_articles.json. Skipping filtering.


True

## Plot graph

In [ ]:
G = create_graph_from_filtered_json()
# Calculation of the disposition (layout) - "Force Atlas"
print('Layout calculation')
try:
    pos = nx.forceatlas2_layout(G,gravity=70,max_iter=150,seed=110)
except Exception:
    pos = nx.spring_layout(G, seed=42)

print("Generation of the figure")
plt.figure(figsize=(15, 8))
nx.draw_networkx(
    G,
    pos,
    node_size=20,
    with_labels=False,
    width=0.05,
    edge_color="#706f6f"
)
plt.title(f"Graph Visualization (Force-Atlas Layout)", fontsize=20)

plt.axis('off')

print('Visualization')
plt.show()
plt.close()

--- Creating graph from filtered JSON (in memory)... ---
Graph created: 75181 nodes, 322519 edges.
Layout calculation


## Community generation

In [4]:
def community_detection():
    results = [] 
    G = create_graph_from_filtered_json()


    # Louvain community detection
    communities = louvain_communities(G, seed=42)

    for comm in communities:
        # Identify the node with the highest degree as the representative
        representative_article = sorted(comm, key=lambda x: G.in_degree(x), reverse=True)[0]
        results.append({
            'representative_node' : representative_article,
            # Convert set to list for JSON serialization
            'community' : list(comm) 
        })

    with open('data/processed/communities.json', 'w') as outfile:
        # Use indent for better JSON readability
        json.dump(results, outfile, indent=4) 

if __name__ == '__main__':
    community_detection()

--- Creating graph from filtered JSON (in memory)... ---
Graph created: 75181 nodes, 322519 edges.


In [11]:
file_path = 'data/processed/communities.json' 

with open(file_path, 'r') as infile:
    commu = json.load(infile)

article_name = {}
with open('data/processed/filtered_articles.json', 'r' ) as infile:
    nodes = json.load(infile)
for article in nodes['articles']:
    article_name[article['id']] = article['title']


In [13]:
for i,commu_i in enumerate(commu):
    print(f"Commu numéro {i+1} : {article_name[commu_i['representative_node']]}")

Commu numéro 1 : Comment on " Low-frequency character of the Casimir force between
  metallic films"
Commu numéro 2 : Fermion-boson duality in integrable quantum field theory
Commu numéro 3 : Evolution of an Atom Impeded by Measurement: The Quantum Zeno Effect
Commu numéro 4 : Observation of an anomalous positron abundance in the cosmic radiation
Commu numéro 5 : The search for differential equations for certain sets of orthogonal
  polynomials
Commu numéro 6 : Physics and Consciousness
Commu numéro 7 : Bounds for the adiabatic approximation with applications to quantum
  computation
Commu numéro 8 : What is the gamma gamma resonance at 750 GeV?
Commu numéro 9 : Primes in the denominators of Igusa Class Polynomials
Commu numéro 10 : Ultra-broadband Heteronuclear Hartmann-Hahn polarization transfer
Commu numéro 11 : The Standard Model Higgs boson as the inflaton
Commu numéro 12 : Implications of a Low sin(2 beta): A Strategy for Exploring New Flavor
  Physics
Commu numéro 13 : Locking e

KeyError: 'quant-ph/0703132'

## Recommandation

In [ ]:
query = ''


In [ ]:
# Core Python libraries
import numpy as np
import pandas as pd
from tqdm import tqdm
import json
import re
import os

# NLP and Embeddings
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
import faiss


# Visualization and evaluation
import matplotlib.pyplot as plt
import seaborn as sns

